In [1]:
!pip install hospital-sim==0.0.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.5/792.5 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

Laster inn API key fra Open Ai. Legg inn navn og value i "Secrets".

In [2]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY er tom.")

os.environ["OPENAI_API_KEY"] = api_key.strip()
print("API-nøkkelen er lastet inn!")

API-nøkkelen er lastet inn!


Når filen kjører, må det filen fra notebook 1 lastes opp her. Trykk "velg filer"

In [4]:
import json
import pandas as pd
from google.colab import files

# Velg JSON-filen dere lastet ned
uploaded = files.upload()
filnavn = next(iter(uploaded))

# Les inn pasientene
pasienter = json.loads(uploaded[filnavn].decode("utf-8"))

# Kontroller innholdet
print(f"Leste inn {len(pasienter)} pasienter")
display(pd.DataFrame(pasienter).head(10))

Saving genererte_pasienter-8.json to genererte_pasienter-8.json
Leste inn 3 pasienter


,profile_id,name,age,gender,chief_complaint,symptoms,medical_history,current_medications,allergies,vital_signs
0,ef8b0963-98a0-463f-b872-fa2af2c05667,Deborah Aita,87,Female,Swelling Knee,"[Swelling Knee, Limited Leg Motion, Fever]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
1,f12afa0e-8847-4b6d-9734-b9984f0770c4,Jessica Stamey,33,Female,Red Eye,"[Red Eye, Watery Eyes, Pain]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
2,ab17de22-eca7-41da-9865-3211411324ce,Wanda Rabinowitz,30,Female,Vomiting,"[Vomiting, Breathing Problems, Fever]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."


Her kjøres simuleringen vha en modifisert hospital-sim kode, som vi har kalt main

In [5]:
import importlib
import main #laster vår egen fil

importlib.reload(main) #leser filen på nytt, slik at endringer blir brukt

# gjør klassene tilgjengelig under samme navn som før
HospitalSimulation = main.HospitalSimulation
Patient = main.Patient

# Opprett en ny simulering
hospital = HospitalSimulation("Colab Emergency Medical Center")

import inspect

# For å kontrollere hvilken fil som faktisk brukes
print("Sykehuset bruker:", inspect.getfile(type(hospital)))
print(
    "Pulsfunksjonen kommer fra:",
    inspect.getfile(type(hospital)._extract_vital_signs)
)

# Lag pasienter fra de tre første profilene i JSON-filen
emergency_patients = []

profil_id_per_pasient = {}

for profil in pasienter[:3]:
    patient = Patient(
        name=profil["name"],
        age=profil["age"],
        gender=profil["gender"],
        chief_complaint=profil["chief_complaint"],
        symptoms=profil["symptoms"],
        medical_history=profil["medical_history"],
        current_medications=profil["current_medications"],
        allergies=profil["allergies"],
        vital_signs=profil["vital_signs"],
    )

    profil_id_per_pasient[patient.patient_id] = profil["profile_id"]

    emergency_patients.append(patient)

# Legg pasientene inn i sykehuset
for patient in emergency_patients:
    hospital.add_patient(patient)

# Kjør simuleringen
hospital.run_simulation(
    duration_minutes=2,
    patient_arrival_rate=0.2
)

2026-10-16 09:56:40 | INFO     | main:_initialize_chromadb:267 - EHR System initialized with ChromaDB collection: hospital_ehr
2026-10-16 09:56:40 | WARNING  | swarms.structs.agent:reliability_check:1921 - Max tokens is set to 128000, but the model 'gpt-4.1-nano' may or may not support 32768 tokens. Please set max tokens to 32768 or less.
2026-10-16 09:56:40 | WARNING  | swarms.structs.agent:reliability_check:1921 - Max tokens is set to 128000, but the model 'o4-mini' may or may not support 100000 tokens. Please set max tokens to 100000 or less.
2026-10-16 09:56:40 | WARNING  | swarms.structs.agent:reliability_check:1921 - Max tokens is set to 128000, but the model 'o4-mini' may or may not support 100000 tokens. Please set max tokens to 100000 or less.
2026-10-16 09:56:40 | WARNING  | swarms.structs.agent:reliability_check:1929 - The model 'groq/deepseek-r1-distill-qwen-32b' may not be supported. Please use a supported model, or override the model name with the 'llm' parameter, which s

╭────────────────────────────────── Agent Name Crystal Moonwhisper [Loop: 1/1] ───────────────────────────────────╮
│ Hello, welcome. I’m Crystal at reception, and I can help get you checked in.                                    │
│                                                                                                                 │
│ To begin, may I have:                                                                                           │
│ - Your full name                                                                                                │
│ - Your age                                                                                                      │
│ - Your chief complaint, or the main reason for today’s visit                                                    │
│                                                                                                                 │
│ If this is your first time here, I can also collect your basic registration information after that.             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── Agent Name Patient_Deborah_Aita [Loop: 1/1] ──────────────────────────────────╮
│ Hello, Crystal. My name is Deborah Aita. I’m 87 years old.                                                      │
│                                                                                                                 │
│ I’m here because my knee is swollen. I’ve also been having trouble moving my leg, and I have a fever. It’s my   │
│ first time here.                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-10-16 09:56:46 | INFO     | main:add_patient:974 - Patient Deborah Aita checked in and added to queue
2026-10-16 09:56:46 | INFO     | main:add_patient:950 - New patient arrival: Jessica Stamey - Red Eye


╭────────────────────────────────── Agent Name Crystal Moonwhisper [Loop: 1/1] ───────────────────────────────────╮
│ Hello, welcome to the clinic. I’m Crystal at the front desk, and I’ll help you get checked in.                  │
│                                                                                                                 │
│ To start, please tell me:                                                                                       │
│ - Full name                                                                                                     │
│ - Age                                                                                                           │
│ - Main reason for today’s visit                                                                                 │
│                                                                                                                 │
│ I also need a few initial details:                                                                              │
│ - Date of birth                                                                                                 │
│ - Phone number                                                                                                  │
│ - Current address                                                                                               │
│ - Insurance provider, if you have one                                                                           │
│ - Whether you’re a new or returning patient                                                                     │
│                                                                                                                 │
│ Once I have that, I can continue your check-in.                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

2026-10-16 09:58:41 | ERROR    | swarms.agents.llm_manager:check_model_supports_utilities:343 - [Agent: Patient_Wanda_Rabinowitz] Model 'groq/deepseek-r1-distill-qwen-32b' does not support function calling capabilities. tools_list_dictionary is set: []. Please use a function calling-enabled model.


╭──────────────────────────────── Agent Name Patient_Wanda_Rabinowitz [Loop: 1/1] ────────────────────────────────╮
│ Hello, Doctor.                                                                                                  │
│                                                                                                                 │
│ I’m having trouble breathing, and that’s the part that worries me the most right now. I don’t have chest pain,  │
│ heavy bleeding, or confusion, and I don’t feel like I’m about to pass out, but I do feel weak and pretty sick   │
│ overall.                                                                                                        │
│                                                                                                                 │
│ I’ve been vomiting, I’ve had a fever, and I’ve been having trouble breathing. It started earlier today and has  │
│ been going on for several hours. It seems to be staying bad, maybe getting a little worse, especially when I    │
│ move around.                                                                                                    │
│                                                                                                                 │
│ I’m not really having pain in one specific place. It’s more that I feel very sick and weak. If I had to rate    │
│ the discomfort, I’d say about a 7 out of 10.                                                                    │
│                                                                                                                 │
│ I’ve definitely had fever. I haven’t really noticed chills. I don’t have a cough or sore throat. I do have      │
│ shortness of breath. I’ve also had nausea and vomiting. I don’t have abdominal pain, diarrhea, burning with     │
│ urination, rash, or neck stiffness. I don’t think I have a headache. I do feel weak, but not numb.              │
│                                                                                                                 │
│ I’m 30 years old. I don’t have any medical conditions that I know of. I haven’t had any surgeries. I’m not      │
│ taking any medications, and I don’t have any known allergies. I don’t smoke, and I don’t use drugs. I drink     │
│ alcohol occasionally.                                                                                           │
│                                                                                                                 │
│ I don’t know of any sick contacts, recent travel, or new exposures. I’m mostly just scared because of the       │
│ breathing trouble and the vomiting.                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-10-16 09:58:46 | ERROR    | swarms.agents.llm_manager:check_model_supports_utilities:343 - [Agent: Dr. Zara Nightingale] Model 'groq/deepseek-r1-distill-qwen-32b' does not support function calling capabilities. tools_list_dictionary is set: []. Please use a function calling-enabled model.


╭────────────────────────────────── Agent Name Dr. Zara Nightingale [Loop: 1/1] ──────────────────────────────────╮
│ Hello, Ms. Rabinowitz.                                                                                          │
│                                                                                                                 │
│ Your ABCs need immediate attention because breathing trouble is part of your presentation:                      │
│ - Airway: you’re able to speak, so airway appears patent                                                        │
│ - Breathing: symptomatic shortness of breath is concerning and needs immediate bedside assessment, pulse        │
│ oximetry, respiratory rate, lung exam, and likely oxygen support if needed                                      │
│ - Circulation: blood pressure 120/80 and heart rate 80 are reassuring, but fever 101.3°F plus weakness and      │
│ vomiting raise concern for significant infection or dehydration                                                 │
│                                                                                                                 │
│ Key clinical features:                                                                                          │
│ - 30 years old                                                                                                  │
│ - Fever 101.3°F                                                                                                 │
│ - Shortness of breath                                                                                           │
│ - Multiple episodes of vomiting                                                                                 │
│ - Generalized weakness                                                                                          │
│ - Symptoms present for several hours and possibly worsening                                                     │
│ - No chest pain                                                                                                 │
│ - No abdominal pain                                                                                             │
│ - No cough or sore throat reported                                                                              │
│ - No known medical history, medications, or allergies                                                           │
│                                                                                                                 │
│ Based on the history alone, I cannot make a single definitive diagnosis without examination and testing, but    │
│ this is an emergency evaluation because of the breathing complaint. My focused exam would include:              │
│ - Respiratory rate and oxygen saturation immediately                                                            │
│ - Work of breathing and ability to speak full sentences                                                         │
│ - Lung exam for wheeze, crackles, or focal findings                                                             │
│ - Cardiac exam                                                                                                  │
│ - Hydration status                                                                                              │
│ - Mental status                                                                                                 │
│ - Abdominal exam despite no pain, given vomiting                                                                │
│ - Skin exam and capillary refill                                                                                │
│                                                                                                                 │
│ Primary working diagnosis:                                                                                      │
│ - Acute febrile illness with shortness of breath and v

2026-10-16 09:58:58 | INFO     | main:add_patient_record:285 - Adding patient record to EHR: Wanda Rabinowitz (ID: 8c1daae6-36a6-43b5-bdbb-8037f8fea9bb) - Doctor: Dr. Zara Nightingale
2026-10-16 09:59:01 | INFO     | main:process_patient_queue:1102 - Patient Wanda Rabinowitz treated by Dr. Zara Nightingale - Diagnosis: Acute febrile illness with shortness of breath and vomiting, primary diagnosis unresolved pending emergency cardiopulmonary and infectious workup - Time: 2.4 minutes
2026-10-16 09:59:02 | INFO     | main:run_simulation:1760 - Simulation ended


In [ ]:
from google.colab import drive
drive.mount('/content/drive')